# 4. Model Deployment
**AAI-540 MLOps Project — Group 6: Student Performance Prediction**

This notebook covers:
- Deploy model as SageMaker Endpoint
- Test endpoint invocation
- Batch Transform job
- Endpoint cleanup

In [ ]:
import pandas as pd
import numpy as np
import joblib
import sagemaker
import boto3
from sagemaker.sklearn import SKLearnModel
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import JSONDeserializer
import warnings
warnings.filterwarnings('ignore')

session = sagemaker.Session()
bucket = session.default_bucket()
prefix = 'student-performance'
role = sagemaker.get_execution_role()

## 4.1 Load Test Data

In [ ]:
test_data = pd.read_csv('../data/processed/test.csv')
X_test = test_data.drop(columns=['performance'])
y_test = test_data['performance']
print(f'Test set: {X_test.shape}')

## 4.2 Deploy SageMaker Endpoint

In [ ]:
# Get model artifact from S3
model_artifact = f's3://{bucket}/{prefix}/model/model.tar.gz'

sklearn_model = SKLearnModel(
    model_data=model_artifact,
    role=role,
    framework_version='1.2-1',
    entry_point='inference.py',
    source_dir='../src/',
    sagemaker_session=session
)

predictor = sklearn_model.deploy(
    initial_instance_count=1,
    instance_type='ml.t2.medium',
    endpoint_name='student-performance-endpoint'
)

print(f'Endpoint deployed: {predictor.endpoint_name}')

## 4.3 Test Endpoint Invocation

In [ ]:
predictor.serializer = CSVSerializer()

# Test with sample data
sample = X_test.iloc[:10].values
result = predictor.predict(sample)

print('=== Endpoint Invocation Results ===')
print(f'Predictions: {result}')
print(f'Actual:      {y_test.values[:10]}')

## 4.4 Batch Transform

In [ ]:
# Save batch input
X_test.to_csv('../data/processed/batch_input.csv', index=False, header=False)

batch_input = session.upload_data(
    path='../data/processed/batch_input.csv',
    bucket=bucket,
    key_prefix=f'{prefix}/batch-input'
)

transformer = sklearn_model.transformer(
    instance_count=1,
    instance_type='ml.m5.large',
    output_path=f's3://{bucket}/{prefix}/batch-output'
)

transformer.transform(
    data=batch_input,
    content_type='text/csv',
    split_type='Line'
)
transformer.wait()
print(f'Batch Transform complete. Output: s3://{bucket}/{prefix}/batch-output')

In [ ]:
# Read batch output
import boto3
s3 = boto3.client('s3')
result = s3.list_objects_v2(Bucket=bucket, Prefix=f'{prefix}/batch-output/')

if 'Contents' in result:
    for obj in result['Contents']:
        print(f'Output file: s3://{bucket}/{obj["Key"]}')
        response = s3.get_object(Bucket=bucket, Key=obj['Key'])
        content = response['Body'].read().decode('utf-8')
        predictions = content.strip().split('\n')[:10]
        print(f'First 10 predictions: {predictions}')

## 4.5 Cleanup — DELETE ENDPOINT
**Run this cell after recording the video demo to avoid charges!**

In [ ]:
# UNCOMMENT AND RUN AFTER VIDEO RECORDING
# predictor.delete_endpoint()
# print('Endpoint deleted.')

## Summary
- Model deployed as SageMaker real-time endpoint
- Endpoint invocation tested with sample data
- Batch Transform job completed
- Cleanup code ready for post-demo